IML 2025 Assignment 2

Jiwoo Hong m12445507  
Yeongjae Park m12446610  
Yoonhyeok Lee m12448710 

## **README**
- **The training and testing will be automatically done if you press 'Run All' button in anaconda jupyter notebook or visual studio code.**  
- **Please install gym, sklearn, tqdm and numpy to run this code.**

import necessary libraries

In [73]:
import gym
import numpy as np
from gym.envs.toy_text.frozen_lake import FrozenLakeEnv, generate_random_map
from tqdm import tqdm
import random


Hyperparameters

In [74]:
MAP_SIZE = 4
MAX_EPISODES = 1
MAX_STEPS = 100
VERBOSE = True


## Define Environment

The envinronment must be satisfy below conditions.
1. is_slippery=true
2. custom maps of different size is allowed
3. the goal state not necessarily in the lower right corner (the start state remains in the upper left corner)
4. other and individual (per tile) probabilities for slipping

So we implemented Customized Environment to satisfy the conditions

for details about 4, the slipping is implemented as like below

ex) Let the probability of slipping of tile that agent is standing on, $p$

if the agent's action is move_left, then the transition probability is

P(move_left) = $1 - p$  
P(move_up) = $\frac{p}{2}$  
P(move_down) = $\frac{p}{2}$  
P(move_right) = $0$  

The definition of "slipping" is written in this page  https://gymnasium.farama.org/environments/toy_text/frozen_lake/#arguments

In [75]:
class CustomizedFrozenLakeEnv(FrozenLakeEnv):
    def __init__(self, desc, map_width, map_height, slip_prob, render_mode=None):
        super().__init__(desc=np.asarray(desc, dtype='c'), is_slippery=False)  # Custom slipping is implemented on stop()
        
        self.desc = np.asarray(desc, dtype='c')
        self.slip_prob: np.array = slip_prob
        self.map_width = map_width
        self.map_height = map_height
        self.render_mode = render_mode
        self.orthogonal_actions = {
            0: [3, 1],  # LEFT -> UP, DOWN
            1: [0, 2],  # DOWN -> LEFT, RIGHT
            2: [3, 1],  # RIGHT -> UP, DOWN
            3: [0, 2]   # UP -> LEFT, RIGHT
        }
        
    def render(self):
        if self.render_mode == "human":
            super().render()
        
    def _get_slip_prob(self, state):
        return self.slip_prob[int(state)]
        
    def step(self, action):
        
        slip_p = self._get_slip_prob(self.s)
        
        action_prob = [0.0, 0.0, 0.0, 0.0]
        action_prob[action] = 1.0 - slip_p
        
        for orthogonal_action in self.orthogonal_actions[action]:
            action_prob[orthogonal_action] += slip_p / 2.0
        
        true_action = np.random.choice(np.arange(4), p=action_prob)
        transitions = self.P[self.s][true_action]
        next_state, reward, done, _ = transitions[0]
        
        self.s = next_state
        self.lastaction = action

        return int(self.s), float(reward), done, {}
    
    def reset(self):
        super().reset()
        self.s = 0
        self.lastaction = None
        return int(self.s), {}

In [76]:


map = None

# Define environment
env = CustomizedFrozenLakeEnv(
    desc=generate_random_map(size=MAP_SIZE),
    map_width=MAP_SIZE,
    map_height=MAP_SIZE,
    slip_prob=np.array([0.1] * (MAP_SIZE * MAP_SIZE)),  # Example slip probabilities
    render_mode='human' if VERBOSE else None
)




ACTION_SPACE = gym.make("FrozenLake-v1").action_space
N_ACTIONS = ACTION_SPACE.n
# N_STATES

## $\epsilon$-greedy

We will implement $\epsilon$-greedy policy with various $\epsilon$ decay.

### Define $\epsilon$-greedy agent

In [77]:
class EpsilonGreedyAgent:
    def __init__(self, epsilon=0.1, decay_rate=0.99, action_space=None, n_states=None, n_actions=None):
        self.epsilon = epsilon
        self.decay_rate = decay_rate
        self.q_table = np.zeros((n_states, n_actions))
        self.action_space = action_space
        
    def select_action(self, state):
        if random.uniform(0, 1) < self.epsilon:
            return self._sample_from_action_space(self.action_space)
        else:
            return np.argmax(self.q_table[state])
        
    def _sample_from_action_space(self, action_space):
        return random.choice(range(self.action_space.n))

#### Training code for $\epsilon$-greedy with decay rate 0

In [81]:
# Run the environment
pbar = tqdm(range(MAX_EPISODES), desc="Episodes", disable=not VERBOSE)
agent = EpsilonGreedyAgent(epsilon=0.1, decay_rate=0.99, action_space=ACTION_SPACE, n_states=MAP_SIZE * MAP_SIZE, n_actions=N_ACTIONS)
for episode in pbar:
    observation, info = env.reset()
    env.render()
    for step in range(MAX_STEPS):
        pbar.set_description(f"Step {step + 1}/{MAX_STEPS}")
        action = agent.select_action(observation)
        observation, reward, done, info = env.step(action)
        if done:
            break
        
env.close()

Step 100/100: 100%|██████████| 1/1 [00:00<00:00,  8.56it/s]


SFFF
FFFF
FFFF
FFFG


TODO  
map customization with specify

more epsilon greedy with various decays
Plot some graphs

DQN (with model explanation)
Plot some graphs

TD(lambda)
Plot some graphs

final analysis

make PDF files for submissions
